In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer, util

In [12]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
import json 

with open("/Users/katharinazeilnhofer/VS Code/ir-project/data_retrieval/kochwiki_data/rezepte_parsed.json", "r") as f:
    recipes = json.load(f)

with open("/Users/katharinazeilnhofer/VS Code/ir-project/data_retrieval/kochwiki_data/zutaten_parsed.json", "r") as f:
    ingredients = json.load(f)

In [14]:
corpus = {}
for i, r in enumerate(recipes):
    doc_id = f"recipe_{i}"
    corpus[doc_id] = {
        "title": r["title"],
        "text": r["plaintext"],
        "type": "recipe"
    }

for i, z in enumerate(ingredients):
    doc_id = f"ingredient_{i}"
    corpus[doc_id] = {
        "title": z["title"],
        "text": z["plaintext"],
        "type": "ingredient"
    }

corpus_texts = [doc["text"] for doc in corpus.values()]
corpus_embeddings = model.encode(corpus_texts)

In [15]:
def se_search(query, n=5):
  query_embedding = model.encode(query)
  scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
  top_indices = scores.argsort(descending=True)[:n]
  doc_id_list = list(corpus.keys())
  return [(doc_id_list[i], corpus[doc_id_list[i]]["title"], scores[i].item()) for i in top_indices]

results = se_search("Tomaten und Basilikum")
for i, (doc_id, title, score) in enumerate(results):
    print(f"Rank {i+1}: [{score:.4f}] {title}")

Rank 1: [0.9584] Geschmolzene Tomaten auf provenzalische Art
Rank 2: [0.9458] Sambal mit Tomate
Rank 3: [0.9359] Zutat:Cherrytomate
Rank 4: [0.9245] Fondue de tomates
Rank 5: [0.9239] Zanderfilet, gebraten auf Tomatencreme


In [16]:
evaluation = {
    "Kartoffelsuppe": ["Vegetarische Kartoffelsuppe", "Vegane Kartoffelsuppe", "Leipziger Kartoffelsuppe", "Leas Kartoffelsuppe", "Lauch-Kartoffelsuppe"],
    "Gulasch": ["Würstchengulasch mit Paprika", "Wurstgulasch", "Winterlicher Krautgulasch", "Wildschweingulasch mit Pilzen und Senfsauce", "Wildschweingulasch mit Pilzen"],
    "Schokoladentorte": ["Himmlische Schokoladentorte", "Schokoladentorte, himmlisch", "Schokoladentorte mit Mascarpone-Minze-Creme", "Maroni-Schokoladentorte", "Brigittes Schokoladentorte"],
    "Weihnachtsplätzchen": ["Walisische Weihnachtsplätzchen", "Waliser Weihnachtsplätzchen", "Detmolder Weihnachtsplätzchen", "Zimtsterne mit Mandeln", "Plätzchen zum Ausstechen"],
    "Brot mit Sauerteig": ["Sauerteigbrot ohne Hefe aus dem Backautomaten", "Sauerteigbrot (hefefrei) aus dem Backautomaten", "Römisches Sauerteigbrot", "Roggenbrot auf Sauerteigbasis", "Kartoffelbrot mit Sauerteig"],
    "japanische Suppe": ["Misosuppe mit Tofu", "Miso-Suppe mit Tofu", "Ramen (Japanische Nudelsuppe)", "Japanische Nudelsuppe", "Miso Shiru"],
    "Curry": ["Tomaten-Linsen-Curry", "Thailändisches Gemüsecurry", "Veganes Blumenkohlcurry", "Thai-Lychee-Curry", "Zanderfilet mit Apfel-Gemüse-Curry"],
    "schnelles Mittagessen mit Nudeln": ["Zitronennudeln", "Studenten-Nudelauflauf", "Tagliatelle in Schinken-Sahnesauce", "Tagliatelle mit Lachs-Sherrysauce", "Zha-Jiang Nudeln nach Peking-Art"],
    "französische Soße": ["Sauce ravigote", "Vinaigrette mit roter Paprikaschote", "Vinaigrette mit Feta", "Erdbeervinaigrette", "Walnusssoße"],
    "Rezepte mit Kartoffeln, Zwiebeln und Speck": ["Bratkartoffeln", "Bauernfrühstück", "Himmel und Erde", "Kartoffelpfanne mit Käse", "Döppekoche"],
}

print("EVALUATION:\n")
total_hits = 0
total_possible = 0

for query, expected in evaluation.items():
    results = se_search(query, n=5)
    retrieved_titles = [corpus[res[0]]["title"] for res in results]
    
    hits = sum(1 for doc in expected if doc in retrieved_titles)
    total_hits += hits
    total_possible += len(expected)
    
    print(f"Query: '{query}'")
    print(f"  Hits: {hits}/5")
    print(f"  Retrieved: {retrieved_titles}")
    print(f"  Expected:  {expected}")
    print()

print(f"{'='*50}")
print(f"Total: {total_hits}/{total_possible} relevant documents found")
print(f"Recall@5: {total_hits/total_possible:.1%}")

EVALUATION:

Query: 'Kartoffelsuppe'
  Hits: 1/5
  Retrieved: ['Vegetarische Kartoffelsuppe', 'Kartoffelsuppe (vegan)', 'Kartoffelsuppen', 'Kartoffelsuppe mit Rauchwurst', 'Kartoffelcremesuppe mti Räucherlachs']
  Expected:  ['Vegetarische Kartoffelsuppe', 'Vegane Kartoffelsuppe', 'Leipziger Kartoffelsuppe', 'Leas Kartoffelsuppe', 'Lauch-Kartoffelsuppe']

Query: 'Gulasch'
  Hits: 0/5
  Retrieved: ['Guavenmarmelade', 'Pörkölt', 'Cacık', 'Gulasch', 'Guacamole (Avocadosauce)']
  Expected:  ['Würstchengulasch mit Paprika', 'Wurstgulasch', 'Winterlicher Krautgulasch', 'Wildschweingulasch mit Pilzen und Senfsauce', 'Wildschweingulasch mit Pilzen']

Query: 'Schokoladentorte'
  Hits: 2/5
  Retrieved: ['Konditorcrème mit Schokolade', 'Schokoladenmousse, leicht', 'Himmlische Schokoladentorte', 'Schokoladekuchen, saftig', 'Schokoladentorte, himmlisch']
  Expected:  ['Himmlische Schokoladentorte', 'Schokoladentorte, himmlisch', 'Schokoladentorte mit Mascarpone-Minze-Creme', 'Maroni-Schokoladentort

In [17]:
queries_with_hit = 0
for query, expected in evaluation.items():
    results = se_search(query, n=5)
    retrieved_titles = [corpus[res[0]]["title"] for res in results]
    if any(doc in retrieved_titles for doc in expected):
        queries_with_hit += 1
        
print(f"{queries_with_hit}/10 queries return at least one relevant document")

4/10 queries return at least one relevant document
